# 1. Dataset Overview

In [61]:
import os
import pandas as pd
#print(os.path.exists("D:\college ml project\individual+household+electric+power+consumption"))
# sep=';' used for sepration in the dataset if not then it is not properly aligned 
df=pd.read_csv("D:\college ml project\individual+household+electric+power+consumption\household_power_consumption.txt",sep=';')
df.head()

C:\Users\adity\AppData\Local\Temp\ipykernel_20268\2248255197.py:5: DtypeWarning: Columns (2,3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv("D:\college ml project\individual+household+electric+power+consumption\household_power_consumption.txt",sep=';')


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0


# 2.No of rows and no of columns to get the overview

In [33]:
df.shape

(2075259, 9)

# The dataset contains 2,075,259 observations and 9 columns.

Each row represents an electricity consumption observation recorded at one-minute intervals.

Date and Time provide temporal information.

The remaining seven columns contain household electricity measurements.

Initial observation: The dataset is large and contains both temporal and numerical information, making it suitable for time-series analysis and forecasting.

# 3.Understanding Data Types

In [36]:
df.dtypes

Date                      object
Time                      object
Global_active_power       object
Global_reactive_power     object
Voltage                   object
Global_intensity          object
Sub_metering_1            object
Sub_metering_2            object
Sub_metering_3           float64
dtype: object

# The initial data type inspection shows:

8 columns have the object data type.

1 column (Sub_metering_3) has the float64 data type.

Date and Time are stored as objects, which is expected before datetime conversion.

The measurement columns are mostly stored as objects because they contain the missing-value marker ?.

Observation: The numerical measurement columns need appropriate type conversion before machine learning.

# 3. Initial Missing-Value Inspection

In [39]:
df.isnull().sum()


Date                         0
Time                         0
Global_active_power          0
Global_reactive_power        0
Voltage                      0
Global_intensity             0
Sub_metering_1               0
Sub_metering_2               0
Sub_metering_3           25979
dtype: int64

# 4.Using isnull().sum(), we identified 25,979 missing values in Sub_metering_3.

However, this does not represent all missing observations in the dataset.

Pandas detects values such as NaN and None as missing values, but does not automatically recognize the string ? as a missing value.

Therefore, a separate inspection is required to identify these markers.

In [45]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2075259 entries, 0 to 2075258
Data columns (total 9 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Date                   object 
 1   Time                   object 
 2   Global_active_power    object 
 3   Global_reactive_power  object 
 4   Voltage                object 
 5   Global_intensity       object 
 6   Sub_metering_1         object 
 7   Sub_metering_2         object 
 8   Sub_metering_3         float64
dtypes: float64(1), object(8)
memory usage: 142.5+ MB


# 4. Identifying Missing-Value Markers

In [52]:
df.eq("?").sum()

Date                         0
Time                         0
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3               0
dtype: int64

# 5.The eq("?").sum() operation identifies the number of question-mark markers in each column.

Observations:

Six measurement columns contain 25,979 question-mark markers each.

Date and Time contain no question-mark markers.

Sub_metering_3 contains no question-mark markers, but has 25,979 actual NaN values.

Conclusion: The dataset contains missing values represented in two different ways: ? and NaN. These need to be standardized during preprocessing.

# 6.Missing Count Percentage


In [96]:
misscountNaN=(df.isnull().sum()/df.shape[0])*100
misscountother=(df.eq("?").sum()/df.shape[0])*100
print(misscountNaN)
print(misscountother)

Date                     0.000000
Time                     0.000000
Global_active_power      0.000000
Global_reactive_power    0.000000
Voltage                  0.000000
Global_intensity         0.000000
Sub_metering_1           0.000000
Sub_metering_2           0.000000
Sub_metering_3           1.251844
dtype: float64
Date                     0.000000
Time                     0.000000
Global_active_power      1.251844
Global_reactive_power    1.251844
Voltage                  1.251844
Global_intensity         1.251844
Sub_metering_1           1.251844
Sub_metering_2           1.251844
Sub_metering_3           0.000000
dtype: float64


# The percentage of missing values is calculated separately for:

Actual missing values (NaN).

Question-mark markers (?).

The results show approximately 1.25% missing observations in the affected measurement columns for each representation.

This analysis helps quantify the extent of missing data before selecting an appropriate treatment strategy.

# Inspecting missing observations

In [80]:
df.loc[:,'Global_active_power'].eq("?").sum()

np.int64(25979)

In [86]:
df.loc[df['Global_active_power']=='?',df.columns].head()

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
6839,21/12/2006,11:23:00,?,?,?,?,?,?,NaN
6840,21/12/2006,11:24:00,?,?,?,?,?,?,NaN
19724,30/12/2006,10:08:00,?,?,?,?,?,?,NaN
19725,30/12/2006,10:09:00,?,?,?,?,?,?,NaN
41832,14/1/2007,18:36:00,?,?,?,?,?,?,NaN


In [93]:
df.loc[df['Global_active_power']=='?']

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
6839,21/12/2006,11:23:00,?,?,?,?,?,?,NaN
6840,21/12/2006,11:24:00,?,?,?,?,?,?,NaN
19724,30/12/2006,10:08:00,?,?,?,?,?,?,NaN
19725,30/12/2006,10:09:00,?,?,?,?,?,?,NaN
41832,14/1/2007,18:36:00,?,?,?,?,?,?,NaN
...,...,...,...,...,...,...,...,...,...
1990185,28/9/2010,19:09:00,?,?,?,?,?,?,NaN
1990186,28/9/2010,19:10:00,?,?,?,?,?,?,NaN
1990187,28/9/2010,19:11:00,?,?,?,?,?,?,NaN
1990188,28/9/2010,19:12:00,?,?,?,?,?,?,NaN


In [94]:
df.loc[6835:6843]

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
6835,21/12/2006,11:19:00,0.244,0.000,242.050,1.000,0.000,0.000,0.0
6836,21/12/2006,11:20:00,0.244,0.000,242.080,1.000,0.000,0.000,0.0
6837,21/12/2006,11:21:00,0.242,0.000,241.670,1.000,0.000,0.000,0.0
6838,21/12/2006,11:22:00,0.244,0.000,242.290,1.000,0.000,0.000,0.0
6839,21/12/2006,11:23:00,?,?,?,?,?,?,NaN
6840,21/12/2006,11:24:00,?,?,?,?,?,?,NaN
6841,21/12/2006,11:25:00,0.246,0.000,241.740,1.000,0.000,0.000,0.0
6842,21/12/2006,11:26:00,0.246,0.000,241.830,1.000,0.000,0.000,0.0
6843,21/12/2006,11:27:00,0.244,0.000,240.960,1.000,0.000,0.000,0.0


# 7.Inspecting Missing Observations
The rows containing missing markers were inspected using Pandas indexing.

The observations show that multiple measurement columns can be missing at the same timestamp.

Some missing observations also occur consecutively.

Initial finding: Missing values may occur in continuous time intervals rather than only as isolated observations.

Further investigation of missing-gap lengths is required before selecting an imputation strategy.